# Ejercicio 05 · Motor recomendador reproducible

Evidencia ejecutada de LAB08/LAB09. MovieLens permanece local; esta libreta resume los reportes reproducibles. Corte temporal por usuario, tres semillas (17, 42, 73).

In [2]:
from pathlib import Path
import pandas as pd, json
root=Path('..') if Path('../reports/main_results.csv').exists() else Path('.')
r=root/'reports'
results=pd.read_csv(r/'main_results.csv'); pareto=pd.read_csv(r/'pareto_comparison.csv'); profiles=pd.read_csv(r/'profile_metrics.csv'); errors=pd.read_csv(r/'error_analysis.csv')
audit=json.loads((r/'data_audit.json').read_text(encoding='utf-8')); cold=json.loads((r/'cold_start_validation.json').read_text(encoding='utf-8'))
print(f"MovieLens: {audit['ratings']:,} valoraciones; {audit['movies']:,} películas; {audit['users']} usuarios")
print(f"Elegibles: {audit['eligible_test_events']}; excluidos por ítem/usuario sin entrenamiento: {audit['excluded_test_events']}")
print('SHA-256 ZIP:',audit['sha256'])

MovieLens: 100,836 valoraciones; 9,742 películas; 610 usuarios
Elegibles: 587; excluidos por ítem/usuario sin entrenamiento: 23
SHA-256 ZIP: 696d65a3dfceac7c45750ad32df2c259311949efec81f0f144fdfb91ebc9e436


## Resultados principales

HitRate@10 usa un único evento futuro por usuario. Cobertura se calcula sobre el catálogo completo. RMSE aplica a la predicción de rating y no sustituye evaluación de ranking.

In [4]:
print(results.round(4).to_string(index=False))

 config            method   rmse  rmse_sd  hit_rate_at_10  hit_rate_sd  catalog_coverage  fit_seconds  model_parameters  evaluated_users
20f_12e     collaborative 1.0272   0.0056          0.0312       0.0055            0.0737      13.5636            206220              587
20f_12e hybrid_alpha_0.25 1.0272   0.0056          0.0290       0.0034            0.1038      13.5636            206220              587
20f_12e hybrid_alpha_0.75 1.0272   0.0056          0.0312       0.0055            0.0760      13.5636            206220              587
 20f_6e     collaborative 1.1005   0.0056          0.0204       0.0074            0.1818       6.8329            206220              587
 20f_6e hybrid_alpha_0.25 1.1005   0.0056          0.0097       0.0064            0.2124       6.8329            206220              587
 20f_6e hybrid_alpha_0.75 1.1005   0.0056          0.0204       0.0061            0.1822       6.8329            206220              587
  8f_6e     collaborative 1.1078   0.0035

## Perfiles y usuarios fríos

Historial corto: hasta el percentil 25 (34 ratings); amplio: desde el percentil 75 (167). La ruta de usuario frío no se puntúa con HitRate temporal porque no tiene historial evaluable.

In [6]:
print(profiles[(profiles.config=='20f_6e') & profiles.method.isin(['collaborative','hybrid_alpha_0.75'])].groupby(['method','profile'],as_index=False).agg(users=('users','first'),hit_rate=('hit_rate_at_10','mean'),coverage=('catalog_coverage','mean')).round(4).to_string(index=False))
print('Validación de fallbacks:',json.dumps(cold,ensure_ascii=False,indent=2))

           method          profile  users  hit_rate  coverage
    collaborative historial_amplio    139    0.0240    0.0397
    collaborative  historial_corto    155    0.0086    0.0965
hybrid_alpha_0.75 historial_amplio    139    0.0240    0.0409
hybrid_alpha_0.75  historial_corto    155    0.0086    0.0955
Validación de fallbacks: {
  "no_history_user": {
    "source": "global_popularity_cold_user",
    "recommendations": [
      "Shawshank Redemption, The (1994)",
      "Godfather, The (1972)",
      "Fight Club (1999)",
      "Star Wars: Episode IV - A New Hope (1977)",
      "Usual Suspects, The (1995)",
      "Godfather: Part II, The (1974)",
      "Princess Bride, The (1987)",
      "Goodfellas (1990)",
      "Schindler's List (1993)",
      "Dark Knight, The (2008)"
    ],
    "count": 10
  },
  "new_user_with_3_known_movies": {
    "source": "content_cold_user_with_history",
    "recommendations": [
      "Chicken Little (2005)",
      "Meet the Robinsons (2007)",
      "Ratch

## Comparación Pareto y decisión

Se consideran utilidad (HitRate@10 y cobertura) y costo (tiempo de ajuste y parámetros). Las opciones no dominadas expresan compromisos distintos, no un ganador universal.

In [8]:
print(pareto.round(4).to_string(index=False))

 config  factors  epochs  alpha   rmse  rmse_sd  hit_rate_at_10  hit_rate_sd  catalog_coverage  fit_seconds  model_parameters  evaluated_users  pareto_efficient
20f_12e       20      12   0.25 1.0272   0.0056          0.0290       0.0034            0.1038      13.5636            206220              587              True
20f_12e       20      12   0.75 1.0272   0.0056          0.0312       0.0055            0.0760      13.5636            206220              587              True
 20f_6e       20       6   0.25 1.1005   0.0056          0.0097       0.0064            0.2124       6.8329            206220              587              True
 20f_6e       20       6   0.75 1.1005   0.0056          0.0204       0.0061            0.1822       6.8329            206220              587              True
  8f_6e        8       6   0.25 1.1078   0.0035          0.0125       0.0020            0.1593       6.7037             82488              587              True
  8f_6e        8       6   0.75 1.

## Auditoría de errores

La tabla de errores contiene los 587 usuarios elegibles para la configuración 8 factores/6 épocas, semilla 17, y registra título retenido, acierto y lista Top-5 para baseline, contenido, colaborativo e híbrido alpha=.75.

In [10]:
print('Filas:',len(errors),'| usuarios:',errors.userId.nunique(),'| aciertos:',int(errors.hit_at_10.sum()))
print(errors.groupby('method').hit_at_10.agg(['count','sum','mean']).round(4).to_string())
print(errors.head(8).to_string(index=False))

Filas: 2348 | usuarios: 587 | aciertos: 34
                   count  sum    mean
method                               
collaborative        587    8  0.0136
content              587    0  0.0000
hybrid_alpha_0.75    587    7  0.0119
popularity           587   19  0.0324
config  seed  userId              profile     method  heldout_movieId                        heldout_title  heldout_rating  hit_at_10  rank_if_hit                                                                                                                                                                               top_recommendations
 8f_6e    17       1     historial_amplio popularity             2492                      20 Dates (1998)             4.0          0          NaN Shawshank Redemption, The (1994) | Godfather, The (1972) | Godfather: Part II, The (1974) | Dark Knight, The (2008) | Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)
 8f_6e    17       2      historial_corto popul

## Límites

Los ratings son autoseleccionados y no miden satisfacción causal. Los metadatos de género son amplios, no hay demografía, y el corte temporal con un evento por usuario tiene alta varianza. Las tres semillas describen estabilidad de inicialización para esta muestra; no prueban significancia estadística ni generalización a producción.